# Voces falsifier v2 — the fragmentation-AND-lexicality-matched non-name-Greek test

The iteration-2 falsifier (`nonname_falsifier`, Cell 10g) compared the voces against short common Greek words
and got **"name-specific"** — but that cohort fragmented ~2× less than the voces (5.15 vs 10.73 Greek
tokens/string) **and** was lexical (real Greek) while the voces are asemic. Three axes at once → uninterpretable.

This notebook fixes it. It builds non-name cohorts **matched to the voces' Greek-token fragmentation** (using the
model's own tokenizer), in both an **asemic** and a **lexical** version, so each axis can be isolated:

- **NAMEHOOD** = voces vs. *asemic non-name, frag-matched* → if ≈0, the deep effect is not about names.
- **FRAGMENTATION** = asemic-matched vs. *original low-frag non-name* → if matched > low-frag, fragmentation drives it.
- **LEXICALITY** = asemic-matched vs. *lexical (real Greek) frag-matched* → if asemic > lexical, meaninglessness matters.

Each deep-gap carries a **bootstrap 95% CI** (the v1 falsifier shipped bare point estimates).

**Run:** Runtime → GPU (T4) → Run all. ~10–15 min on Mistral-7B-v0.3 (4-bit). Downloads one results JSON.

In [ ]:
# --- deps ---
%pip -q install "transformers>=4.44" accelerate bitsandbytes sentencepiece 2>/dev/null
import torch, numpy as np, random
SEED=0; np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# --- model (Mistral-7B-v0.3, 4-bit on T4; no HF token needed) ---
MODEL_NAME = "mistralai/Mistral-7B-v0.3"   # point at "google/gemma-2-9b" (+HF_TOKEN) to also resolve the Gemma anomaly
LOAD_4BIT  = True
VERSION    = "falsifier-v2-lexicality_" + MODEL_NAME.split("/")[-1]

import os
HF_TOKEN = os.environ.get("HF_TOKEN","")
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or (userdata.get("HF_TOKEN") or "")
except Exception: pass
if HF_TOKEN:
    from huggingface_hub import login; login(HF_TOKEN); print("HF: logged in")

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None: tok.pad_token = tok.eos_token
kw = dict(torch_dtype=torch.float16, device_map="auto", output_hidden_states=True, low_cpu_mem_usage=True)
if "gemma-2" in MODEL_NAME.lower(): kw["attn_implementation"]="eager"
if LOAD_4BIT:
    kw["quantization_config"]=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kw).eval()
N_LAYERS = model.config.num_hidden_layers; D_MODEL = model.config.hidden_size
DEV = next(model.parameters()).device
print("loaded:", MODEL_NAME, "| layers:", N_LAYERS, "| device:", DEV)

In [ ]:
# --- Betz Latin -> uppercase Greek transliterator (IDENTICAL to voces_crossfamily.ipynb) ---
_DIGRAPHS=[("TH","Θ"),("PH","Φ"),("CH","Χ"),("PS","Ψ"),("NG","ΝΓ")]
_SINGLES={"A":"Α","B":"Β","G":"Γ","D":"Δ","E":"Ε","Z":"Ζ","H":"Η","I":"Ι","K":"Κ","L":"Λ","M":"Μ",
          "N":"Ν","X":"Ξ","O":"Ο","P":"Π","R":"Ρ","S":"Σ","T":"Τ","U":"Υ","Y":"Υ","W":"Ω"}
def to_greek(s):
    s=s.upper()
    for a,b in _DIGRAPHS: s=s.replace(a,b)
    return "".join(_SINGLES.get(c,c) for c in s)
def ngreek_tok(latin):  # Greek-token count of a latin string's Greek rendering, this model's tokenizer
    return len(tok.encode(to_greek(latin), add_special_tokens=False))

In [ ]:
# --- carrier-frame extraction (IDENTICAL harness to voces_crossfamily.ipynb) ---
FRAME_PRE="The string "; FRAME_POST=" is written on the page."
@torch.no_grad()
def extract_reps(s):
    pre_ids=tok(FRAME_PRE, add_special_tokens=True).input_ids
    full_ids=tok(FRAME_PRE+s, add_special_tokens=True).input_ids
    start=len(pre_ids); end=len(full_ids)
    ids=tok(FRAME_PRE+s+FRAME_POST, return_tensors="pt", add_special_tokens=True).input_ids.to(DEV)
    end=min(end, ids.shape[1])
    if end<=start: start=max(0,end-1)
    hs=model(ids).hidden_states
    return np.stack([h[0,start:end,:].float().mean(0).cpu().numpy() for h in hs])   # [L+1, d]
def build_matrix(strings):
    return np.stack([extract_reps(s) for s in strings])   # [n, L+1, d]
print("harness ready.")

In [ ]:
# --- anchors + voces (latin forms; transliterate to Greek with to_greek) ---
NAMES_LAT=["AGAMEMNON","ODYSSEUS","ACHILLEUS","PENELOPE","ALEXANDROS","SOKRATES","ARISTOTELES",
           "HERAKLES","PERSEPHONE","APHRODITE","POSEIDON","ARTEMIS","DIONYSOS","ASKLEPIOS","HEPHAISTOS","KLEOPATRA"]
_CONS=list("BGDZKLMNPRSTFCH"); _VOW=list("AEIOUY")
def rand_cons(n): return "".join(random.choice(_CONS) for _ in range(n))
RANDOM_LAT=[rand_cons(random.randint(5,11)) for _ in range(28)]

# non-theonym, pure-asemic voces (the low-T arm) — latin forms
VOCES_LAT=["ABLANATHANALBA","AKRAMMACHAMAREI","SESENGENBARPHARANGES","SEMESILAM","SEMESEILAMPS","MASKELLI",
 "MASKELLO","NEBOUTOSOUALETH","DAMNAMENEUS","ASKION","KATASKION","LIXTETRAX","AISION","BAINCHOOOCH","THOBARRABAU",
 "CHABRACH","PHNESCHER","BORKA","PHORBA","AROURARELYOTH","KMEPH","THENOR","AEEIOYO","IEOUAEOI","OREOBAZAGRA",
 "PROTOPHANES","NEPHERIERI","AKTIOPHI","ORORIOUTH","BAKAXICHYCH","ABERAMENTHOOU","LERTHEXANAX","SOROORMERPHERGAR",
 "PATATHNAX","IARBATHA","SANKANTHARA","PSINOTHER","CHTHETHONI","MENEBAINCHUCH","BARZAN","SISISRO","THENOB",
 "ARSENOPHRE","BACHUCH","PHORPHORBA","HARMONTHARTHOCHE","NEOPHOXOTHA","ARISTANABAZAO","PCHORBAZANACHU",
 "ZALAMOIRLALITH","EILESILARMU","TANTINURACHTH","CHORCHORNATHI","PHANTHENPHYPHLIA","AZAZAEISTHAILICH",
 "MENNYTHYTH","SERYCHARRALMIO","AKHEBUKROM"]
THEONYM_ROOTS=["IAO","SABAOTH","BAOTH","AOTH","ADONAI","ELOAI","ELOI","ABRASAX","ABRAXAS","ABRA","MICHA",
 "ERESHKIGAL","SABA","ADON","OTH","IAE","IEOU","SETH","HOR","OSIR","ISIS","AMOUN","THOTH"]
def contamination_T(s):
    s=s.upper(); cov=[False]*len(s)
    for r in THEONYM_ROOTS:
        st=0
        while True:
            i=s.find(r,st)
            if i<0: break
            for j in range(i,i+len(r)): cov[j]=True
            st=i+1
    return sum(cov)/max(1,len(s))
VOCES_LOWT=[v for v in VOCES_LAT if contamination_T(v)<0.15]
TARGET = int(round(np.median([ngreek_tok(v) for v in VOCES_LOWT])))   # voces median Greek tokens
print(f"voces low-T n={len(VOCES_LOWT)} | median Greek tok/str (TARGET) = {TARGET}")

In [ ]:
# --- ORIGINAL low-frag non-name cohort (replicates the confounded v1 baseline) ---
NONNAME_LOWFRAG=["EN","DYO","TRIA","TESSARA","PENTE","EX","EPTA","OKTO","ENNEA","DEKA","KAI","ALLA","OUTOS",
 "GAR","MEN","EIS","EK","PROS","META","PERI","BIBLION","KARDIA","LOGOS","ERGON","TOPOS","PHONE"]

# --- LEXICAL non-name cohort: REAL long Greek words (latin Betz form -> to_greek = real Greek), frag-matched ---
LEXICAL_POOL=["ANTHROPOLOGIA","PHILOSOPHIA","DEMOKRATIA","EPISTEMOLOGIA","BIBLIOTHEKE","KATASTROPHE",
 "METAMORPHOSIS","PARASKEUE","OIKONOMIA","GEOGRAPHIA","ASTRONOMIA","MATHEMATIKE","GRAMMATIKE","ARCHITEKTONIKE",
 "THEOLOGIA","PSYCHOLOGIA","TECHNOLOGIA","PHILANTHROPIA","ENKYKLOPAIDEIA","ARISTOKRATIA","DIALEKTIKE",
 "METAPHYSIKA","KOSMOLOGIA","ETYMOLOGIA","PARADEIGMATA","SYMPHONIA","EUCHARISTIA","PROPHETEIA","KATHOLIKOS",
 "OIKOUMENIKOS","HERMENEUTIKE","PALAIONTOLOGIA","CHARAKTERISTIKOS","PARTHENOGENESIS","ELEUTHERIA"]
def near_target(latin, tol=2): return abs(ngreek_tok(latin)-TARGET)<=tol
NONNAME_LEXICAL=[w for w in LEXICAL_POOL if near_target(w)]
print(f"lexical frag-matched: {len(NONNAME_LEXICAL)} words near {TARGET}±2 Greek tok")
print("  e.g.", [(w, ngreek_tok(w)) for w in NONNAME_LEXICAL[:6]])

# --- ASEMIC frag-matched non-name cohort: random latin asemic, length-tuned to TARGET Greek tokens ---
def gen_asemic_matched(n, tol=1, max_tries=20000):
    out=[]; tries=0
    while len(out)<n and tries<max_tries:
        tries+=1
        L=random.randint(6,14)
        s="".join(random.choice(_CONS+_VOW) for _ in range(L))
        if abs(ngreek_tok(s)-TARGET)<=tol: out.append(s)
    return out
NONNAME_ASEMIC=gen_asemic_matched(28)
print(f"asemic frag-matched: {len(NONNAME_ASEMIC)} strings at {TARGET}±1 Greek tok")
print("  e.g.", [(w, ngreek_tok(w)) for w in NONNAME_ASEMIC[:6]])

In [ ]:
# --- fragmentation sanity table (the whole point: Greek tok/str per cohort) ---
def cohort_frag(latins):
    a=np.array([ngreek_tok(x) for x in latins]); return float(a.mean()), float(np.median(a)), len(a)
for nm, c in [("voces_lowT",VOCES_LOWT),("nonname_asemic_MATCHED",NONNAME_ASEMIC),
              ("nonname_lexical_MATCHED",NONNAME_LEXICAL),("nonname_lowfrag",NONNAME_LOWFRAG)]:
    m,md_,n=cohort_frag(c); print(f"  {nm:26s} Greek tok/str mean={m:5.2f} median={md_:4.1f} (n={n})")

In [ ]:
# --- extract residual streams for all cohorts, both scripts ---
import time; t0=time.time()
COHORTS={"name":NAMES_LAT,"random":RANDOM_LAT,"voces":VOCES_LOWT,
         "nn_asemic":NONNAME_ASEMIC,"nn_lexical":NONNAME_LEXICAL,"nn_lowfrag":NONNAME_LOWFRAG}
REPS={"latin":{},"greek":{}}
for k,latins in COHORTS.items():
    REPS["latin"][k]=build_matrix(latins)
    REPS["greek"][k]=build_matrix([to_greek(x) for x in latins])
    print(f"  {k:12s} latin{REPS['latin'][k].shape} greek{REPS['greek'][k].shape}")
print("extraction done in", round(time.time()-t0,1),"s")

In [ ]:
# --- deep Greek-minus-Latin name-likeness, with bootstrap 95% CI (addresses 'bare point estimate') ---
DEEP=range(int(0.5*N_LAYERS), N_LAYERS+1)
def _nrm(x): return x/(np.linalg.norm(x)+1e-9)
def _nl_per_string(cohort_key, script, idx):
    # name-likeness of one string at each layer, vs this script's name/random centroids
    A=REPS[script][cohort_key]
    nc=lambda L: REPS[script]['name'][:,L,:].mean(0)
    rc=lambda L: REPS[script]['random'][:,L,:].mean(0)
    return np.array([_nrm(A[idx,L,:])@_nrm(nc(L)) - _nrm(A[idx,L,:])@_nrm(rc(L)) for L in range(N_LAYERS+1)])
def deep_gap_per_string(cohort_key):
    n=REPS['latin'][cohort_key].shape[0]
    g=np.array([np.mean([_nl_per_string(cohort_key,'greek',i)[L]-_nl_per_string(cohort_key,'latin',i)[L] for L in DEEP]) for i in range(n)])
    return g   # per-string deep gap
def boot_ci(per_string, B=2000):
    n=len(per_string); rng=np.random.RandomState(SEED)
    means=[per_string[rng.randint(0,n,n)].mean() for _ in range(B)]
    return float(per_string.mean()), float(np.percentile(means,2.5)), float(np.percentile(means,97.5))

GAPS={}
for k in ["voces","nn_asemic","nn_lexical","nn_lowfrag"]:
    ps=deep_gap_per_string(k); m,lo,hi=boot_ci(ps)
    GAPS[k]=dict(mean=m, ci_lo=lo, ci_hi=hi, n=len(ps))
    print(f"  {k:12s} deep gap {m:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  (n={len(ps)})")

In [ ]:
# --- the factorial verdict ---
def diff_ci(a,b,B=2000):  # bootstrap CI on (gap_a - gap_b), independent resamples
    pa=deep_gap_per_string(a); pb=deep_gap_per_string(b); rng=np.random.RandomState(SEED+1)
    d=[pa[rng.randint(0,len(pa),len(pa))].mean()-pb[rng.randint(0,len(pb),len(pb))].mean() for _ in range(B)]
    return float(pa.mean()-pb.mean()), float(np.percentile(d,2.5)), float(np.percentile(d,97.5))
def verdict(name, a, b, hi_means_a):
    d,lo,hi=diff_ci(a,b); sig = (lo>0 or hi<0)
    print(f"\n[{name}]  {a} − {b} = {d:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  {'SIGNIFICANT' if sig else 'n.s. (CI spans 0)'}")
    return dict(diff=d, ci_lo=lo, ci_hi=hi, significant=bool(sig))

print("="*72); print("FACTORIAL FALSIFIER — model:", MODEL_NAME); print("="*72)
R={}
R["NAMEHOOD"]      = verdict("NAMEHOOD: are names special at matched frag+lexicality?", "voces","nn_asemic", True)
R["FRAGMENTATION"] = verdict("FRAGMENTATION: does asemic Greek persist more when more fragmented?", "nn_asemic","nn_lowfrag", True)
R["LEXICALITY"]    = verdict("LEXICALITY: does asemic persist more than meaningful Greek at matched frag?", "nn_asemic","nn_lexical", True)

print("\n--- reading ---")
print("NAMEHOOD n.s.  -> deep effect is NOT about names (voces ≈ frag-matched asemic non-names)")
print("FRAGMENTATION + -> fragmentation drives the deep-Greek persistence (the paper's thesis)")
print("LEXICALITY +   -> asemic-ness/meaningfulness ALSO matters (a real second factor)")

In [ ]:
# --- save results JSON (download from the Files pane) ---
import json
TARGET_FRAG={k:cohort_frag(c) for k,c in [("voces",VOCES_LOWT),("nn_asemic",NONNAME_ASEMIC),
            ("nn_lexical",NONNAME_LEXICAL),("nn_lowfrag",NONNAME_LOWFRAG)]}
out=dict(model=MODEL_NAME, version=VERSION, quantized=bool(LOAD_4BIT), n_layers=int(N_LAYERS),
         seed=SEED, target_greek_tok=TARGET,
         cohort_fragmentation={k:dict(mean=v[0],median=v[1],n=v[2]) for k,v in TARGET_FRAG.items()},
         deep_gaps=GAPS, factorial=R)
fn=f"voces_{VERSION}_results.json"
json.dump(out, open(fn,"w"), indent=2)
print(json.dumps(out, indent=2))
print("\nsaved:", fn)
try:
    from google.colab import files; files.download(fn)
except Exception: pass